# Verify Model Feature Names

Load the registered model from GCS and inspect the feature names it was trained on.
Use this to confirm the model expects canonical feature names (`sepal_length_cm`, etc.)
after retraining with the feature store pipeline.

## Config

In [ ]:
PROJECT = "deeplearning-sahil"
REGION = "us-central1"
MODEL_NAME = "Iris-Classifier-XGBoost-staging"

## 1. Find the latest registered model and its artifact URI

In [ ]:
from google.cloud import aiplatform_v1

client = aiplatform_v1.ModelServiceClient(
    client_options={"api_endpoint": f"{REGION}-aiplatform.googleapis.com"}
)

parent_models = list(client.list_models(
    request={
        "parent": f"projects/{PROJECT}/locations/{REGION}",
        "filter": f"display_name={MODEL_NAME}",
    }
))
parent_models.sort(key=lambda m: m.create_time, reverse=True)

model = parent_models[0]
print(f"Model: {model.display_name}")
print(f"Created: {model.create_time}")
print(f"Artifact URI: {model.artifact_uri}")

## 2. Download and load model.joblib from GCS

In [ ]:
import fsspec
import joblib

model_uri = model.artifact_uri.rstrip("/") + "/model.joblib"
print(f"Downloading: {model_uri}")

fs, _ = fsspec.core.url_to_fs(model_uri)
with fs.open(model_uri, "rb") as f:
    xgb_model = joblib.load(f)

print(f"Model type: {type(xgb_model).__name__}")

## 3. Inspect feature names the model was trained on

In [ ]:
print("Feature names from training (feature_names_in_):")
print(list(xgb_model.feature_names_in_))
print(f"\nNumber of features: {xgb_model.n_features_in_}")

expected = ["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]
actual = list(xgb_model.feature_names_in_)

if actual == expected:
    print("\n✅ Feature names match canonical names from feature platform")
else:
    print(f"\n❌ Feature name mismatch!")
    print(f"  Expected: {expected}")
    print(f"  Actual:   {actual}")

## 4. Test prediction with canonical feature names

In [ ]:
import pandas as pd

sample = pd.DataFrame([{
    "sepal_length_cm": 5.1,
    "sepal_width_cm": 3.5,
    "petal_length_cm": 1.4,
    "petal_width_cm": 0.2,
}])

prediction = xgb_model.predict(sample)
probabilities = xgb_model.predict_proba(sample)

species_map = {0: "Iris-versicolor", 1: "Iris-virginica", 2: "Iris-setosa"}

print(f"Prediction: {int(prediction[0])} ({species_map.get(int(prediction[0]), 'unknown')})")
print(f"Probabilities: {probabilities[0].tolist()}")